# Global Stuffs

In [1]:
# use env bl17_2 to run this script

import param
import panel as pn

import holoviews as hv
import imageio
import os, glob, time

import json
import pandas as pd

import tqdm
import numpy as np


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

from tkinter import font
import numpy as np
import matplotlib.pyplot as plt
import pyFAI

import numpy as np
import pyFAI
import os
import matplotlib.pyplot as plt

# Functions

In [2]:
def set_plot_style(axs, fonts, xlabel, ylabel):
    axs.set_xlabel(xlabel, fontsize=fonts)
    axs.set_ylabel(ylabel, fontsize=fonts)
    axs.tick_params(axis='both', which='major', direction='out', length=4, width=1)
    axs.tick_params(which='minor', width=1, size=2)  # Adjust size as needed
    axs.minorticks_on() # Turn on minor ticks
    
    #axs.grid(False, which='both', axis='both', linestyle='--', linewidth=0.5)
    axs.set_facecolor('white')
    axs.spines['top'].set_linewidth(1)
    axs.spines['right'].set_linewidth(1)
    axs.spines['bottom'].set_linewidth(1)
    axs.spines['left'].set_linewidth(1)
    axs.tick_params(axis='x', labelsize=fonts)
    axs.tick_params(axis='y', labelsize=fonts)

    return axs

## save subtracted data

In [3]:
import os
import pandas as pd

def save_subtracted_txt(q, I_sub, I_sub_sigma, I_raw, fname_sample, fname_bkg, fname_empty, timer, bstop, ctemp, I0, out_dir):
    """
    Save subtracted WAXS data to a .dat text file with headers.

    Parameters:
    - q: array-like, q values in nm^-1
    - I_sub: array-like, background-subtracted intensity
    - I_sub_sigma: array-like, sigma of subtracted intensity
    - I_raw: array-like, raw or normalized sample intensity
    - fname_sample: str, sample filename
    - fname_bkg: str, background filename
    - fname_empty: str, empty scan filename
    - timer: float or str, exposure time
    - bstop: float or str, beamstop position
    - ctemp: float or str, camera temperature
    - I0: float or str, incident beam intensity
    - out_dir: str, output directory
    """

    # Create DataFrame
    data = {
        "q_nm^-1": q,
        "I_avg_sub": I_sub,
        "I_avg_sub_sigma": I_sub_sigma,
        "I_sample": I_raw,
    }
    df = pd.DataFrame(data)

    # Output file name
    out_file = os.path.join(out_dir, f"{fname_sample.split('.dat')[0]}_sub.dat")

    # Header lines
    inst_info = f"Timer: {timer}, bstop: {bstop}, ctemp: {ctemp}, I0: {I0}"
    headers = [
        f"filename: {fname_sample}",
        f"background : {fname_bkg}",
        f"Empty : {fname_empty}",
        inst_info,
        "fit_data",
        "q_nm^-1 ------ I_avg_subtracted ------ I_avg_subtracted_sigma ------ I_Normalized",
    ]
    commented = ['# ' + line for line in headers]

    # Write file
    with open(out_file, 'w') as f:
        f.write('\n'.join(commented) + '\n')
        df.to_csv(f, sep='\t', index=False, header=False)


## read HDF5 file


In [4]:
def read_h5_data(base_path, samp_folder, keyword, azimuthal=True):
    """
    Read azimuthal or radial data from HDF5 files generated by integrate_tif_files.

    Parameters:
    -----------
    base_path : str
        Root path to the data.
    samp_folder : str
        Folder name corresponding to the sample.
    keyword : str
        Keyword like "PS_500C" etc.
    azimuthal : bool, optional
        If True, reads azimuthal data; otherwise reads radial data.

    Returns:    
    --------
    data_list : list of dict
        Each entry corresponds to one dataset stored in the HDF5 file.
    """
    import h5py
    import os
    import glob

    # Construct the expected folder path
    target_folder = os.path.join(base_path, samp_folder)
    if not os.path.exists(target_folder):
        raise FileNotFoundError(f"Target folder does not exist: {target_folder}")

    # Locate HDF5 files
    pattern = f"*{keyword}*_azimuthal_data.h5" if azimuthal else f"*{keyword}*_radial_data.h5"
    h5_files = glob.glob(os.path.join(target_folder, pattern))

    if not h5_files:
        raise FileNotFoundError(f"No matching HDF5 files found in {target_folder} with pattern {pattern}")

    data_list = []

    for h5_path in h5_files:
        with h5py.File(h5_path, 'r') as hf:
            for key in hf:
                group = hf[key]

                def get_safe_str(name):
                    val = group.get(name)
                    if val is not None:
                        val = val[()]
                        return val.decode() if isinstance(val, bytes) else val
                    return "N/A"

                def get_safe_val(name):
                    val = group.get(name)
                    return val[()] if val is not None else None

                data = {
                    "filename": get_safe_str("filename"),
                    "real_date_time": get_safe_str("real_date_time"),
                    "i0": get_safe_val("i0"),
                    "i1": get_safe_val("i1"),
                    "Photod": get_safe_val("Photod")
                }

                if azimuthal:
                    data.update({
                        "q": group["q"][()],
                        "I": group["I"][()],

                    # erro in not presetn then detmine it by sqrt of I
                        "error": group["error"][()] if "error" in group else np.sqrt(group["I"][()]),
            

                        
                    })
                else:
                    data.update({
                        "chi": group["chi"][()],
                        "I": group["I"][()]
                    })

                data_list.append(data)
        
        # print real date and time
        for d in data_list:
            print(d["real_date_time"])

    return data_list


## write HDF5 file

In [5]:
import h5py
import numpy as np

def write_sorted_data_to_h5(sorted_data, output_file, mode='unknown'):
    """
    Write sorted peak data into an HDF5 file.

    Parameters:
    -----------
    sorted_data : list of tuples
        Each tuple should contain (peak_intensity, x_array, I_array, filename, scan_number).
        Missing values will be replaced with default 0 or empty array.
    output_file : str
        Full path where the HDF5 file will be saved.
    mode : str, optional
        Type or mode to store in the dataset ('radial', 'azimuthal', etc.). Default is 'unknown'.
    """
    with h5py.File(output_file, 'w') as hf:
        for idx, entry in enumerate(sorted_data):
            # Fill missing values with defaults
            peak = entry[0] if len(entry) > 0 else 0
            x = np.array(entry[1]) if len(entry) > 1 else np.array([0])
            I = np.array(entry[2]) if len(entry) > 2 else np.array([0])
            filename = str(entry[3]) if len(entry) > 3 else 'unknown'
            scan_number = entry[4] if len(entry) > 4 else 0

            group = hf.create_group(f'data_{idx}')
            group.create_dataset('x', data=x)
            group.create_dataset('I', data=I)
            group.create_dataset('filename', data=filename)
            group.create_dataset('peak_intensity', data=peak)
            group.create_dataset('type', data=str(mode))
            group.create_dataset('scan_number', data=scan_number)
            group.create_dataset('background_filename', data=filename)
            group.create_dataset('background_file', data=filename)

    print(f"✅ Successfully wrote {len(sorted_data)} entries to '{output_file}'.")



## subtract function v2

In [6]:
import os
from matplotlib import scale
import numpy as np
import matplotlib.pyplot as plt
import h5py

def subtract_background_v2(
    base_path,
    samp_folder,
    bkg_folder,
    keyword,
    background_keyword,
    subtract_azimuthal=True,
    subtract_chi=False,
    scaling_factor=1.0,
    plot=True,
    use_sample_as_background=False,
    background_filename=None,
    match_partial=False
):
    def process_subtraction(sample_data, background_data, x_key, save_suffix):
        subtracted_data = []

        for samp, bg in zip(sample_data, background_data):
            print(f"Subtracting background from sample:")
            print(f"  Sample file     : {samp['filename']}")
            print(f"  Background file : {bg['filename']}\n")

            x = samp[x_key]
            I_sample = np.array(samp["I"])


            if subtract_azimuthal and "error" in samp:
                err = samp["error"]
            else:
                err = None

            # find the average of I values in the range of 8-10 q range
            if x_key == "q":
                q_range = (x >= 4) & (x <= 7)
                if np.any(q_range):
                    avg_I = np.mean(I_sample[q_range])
                    avg_I_bg = np.mean(bg["I"][q_range])
                    print(f"Average I in q range 8-10: {avg_I}")
                    print(f"Average I in background q range 8-10: {avg_I_bg}")
                    if avg_I_bg > 0:
                        scaling_factor = 0 # 0.98*(avg_I / avg_I_bg)
                        print(f"Scaling factor for azimuthal data: {scaling_factor}")
                    else:
                        print("Warning: Average I in background is zero, using default scaling factor.")
                        scaling_factor = 0 #1.0
                else:
                    print("Warning: No valid q range found for scaling.")
                    scaling_factor = 0 #1.0
            
            else:
                # For chi data, we can use a different range or method
                chi_range = (x >= -3) & (x <= 0)
                if np.any(chi_range):
                    avg_I = np.mean(I_sample[chi_range])
                    avg_I_bg = np.mean(bg["I"][chi_range])
                    print(f"Average I in chi range 8-10: {avg_I}")
                    print(f"Average I in background chi range 8-10: {avg_I_bg}")
                    if avg_I_bg > 0:
                        scaling_factor = 0 #*(avg_I / avg_I_bg)
                        print(f"Scaling factor for chi data: {scaling_factor}")
                    else:
                        print("Warning: Average I in background is zero, using default scaling factor.")    
                        # Use the average of the sample data for scaling
                        #print(f"Scaling factor for azimuthal data: {scaling_factor}")
                        scaling_factor = 0
                else:
                    print("Warning: No valid q range found for scaling.")
                    scaling_factor = 0
            I_background = np.array(bg["I"]) * scaling_factor

            real_date_time = samp["real_date_time"]
            i0 = samp["i0"]
            i1 = samp["i1"]
            Photod = samp["Photod"]

            I_subtracted = I_sample - I_background


            
            subtracted_data.append({
                x_key: x,
                "I": I_subtracted,
                "filename": samp["filename"],
                "err": err,
                "real_date_time": samp["real_date_time"],  # store per-sample
                "i0": samp["i0"],
                "i1": samp["i1"],
                "Photod": samp["Photod"],
            })


            if plot:
                plt.figure(figsize=(6, 4))
                plt.plot(x, I_sample, '-', markersize=1, label='Sample')
                plt.plot(x, I_background, '-', markersize=1, label='Background')
                plt.plot(x, I_subtracted, '-', markersize=1, label='Subtracted')
                plt.xlabel(x_key + ' [1/nm]' if x_key == "q" else 'Chi [degrees]')
                plt.ylabel('Intensity [a.u.]')
                plt.title(f"Subtraction: {samp['filename']}")
                set_plot_style(plt.gca(), 20, x_key + ' [1/nm]' if x_key == "q" else 'Chi [degrees]', 'Intensity [a.u.]')
                plt.legend()
                plt.tight_layout()
                plt.show()

        # Save to HDF5
        save_folder = os.path.join(base_path)
        output_folder = os.path.join(save_folder, 'Subtracted_Data')
        os.makedirs(output_folder, exist_ok=True)
        output_file = os.path.join(output_folder, f'subtracted_{keyword}_{save_suffix}.h5')
        if os.path.exists(output_file):
            print(f"File already exists: {output_file}. Overwriting.")
        else:
            print(f"Creating new file: {output_file}")
        
        print(f"Saving subtracted data to {output_file}")

        with h5py.File(output_file, 'w') as hf:
            for idx, data in enumerate(subtracted_data):
                group = hf.create_group(f'data_{idx}')
                
                # Determine x_key: either 'q' or 'chi'
                x_key = 'q' if 'q' in data else 'chi'
                
                # Save both 'x' (generic) and specific axis ('q' or 'chi')
                group.create_dataset('x', data=data[x_key])
                group.create_dataset(x_key, data=data[x_key])
                # Replace 'azimuthal' with this in HDF5 writing section:
                if x_key == 'q' and 'err' in data:
                    group.create_dataset('error', data=data['err'])
                
                group.create_dataset('I', data=data["I"])
                group.create_dataset('filename', data=data["filename"])
                group.create_dataset('scaling_factor', data=scaling_factor)
                group.create_dataset('type', data=save_suffix)
                group.create_dataset('background_filename', data=background_filename if use_sample_as_background else background_keyword)
                group.create_dataset('background_file', data=background_data[idx]['filename'])
                group.create_dataset('real_date_time', data=data["real_date_time"])
                group.create_dataset('i0', data=np.float32(data["i0"]) if data["i0"] is not None else np.nan)
                group.create_dataset('i1', data=np.float32(data["i1"]) if data["i1"] is not None else np.nan)
                group.create_dataset('Photod', data=np.float32(data["Photod"]) if data["Photod"] is not None else np.nan)


        return subtracted_data

    all_results = {}

    for mode, x_key, flag in [
        ("azimuthal_data", "q", subtract_azimuthal),
        ("radial_data", "chi", subtract_chi)
    ]:
        if not flag:
            continue

        # Read sample data
        sample_data = read_h5_data(base_path, samp_folder, keyword, azimuthal=(x_key == "q"))
        sample_data = sorted(sample_data, key=lambda x: x['filename'])

        if use_sample_as_background:
            if background_filename is None:
                raise ValueError("Please specify 'background_filename' when using a sample file as background.")

            selected_bg = None
            for data in sample_data:
                fname = data["filename"]
                if (fname == background_filename) or (match_partial and background_filename in fname):
                    selected_bg = data
                    break

            if selected_bg is None:
                print("Available filenames:")
                for d in sample_data:
                    print(" -", d["filename"])
                raise ValueError(f"No sample file matches background_filename='{background_filename}'.")

            background_data = [selected_bg] * len(sample_data)
            print(f"\n✅ Using sample file as background: {selected_bg['filename']} for {mode}\n")
        else:
            background_data = read_h5_data(base_path, bkg_folder, background_keyword, azimuthal=(x_key == "q"))
            background_data = sorted(background_data, key=lambda x: x['filename'])

            if len(sample_data) != len(background_data):
                # use first background file for all
                print(f"Sample data length: {len(sample_data)}")
                print(f"Background data length: {len(background_data)}")
                print("Using first background file for all samples.")
                background_data = [background_data[0]] * len(sample_data)
                #raise ValueError(f"Sample and background data lengths do not match for {mode} subtraction.")

        all_results[mode] = process_subtraction(sample_data, background_data, x_key, save_suffix=mode)

    return all_results





# Subtract background 

# inputs: Reduced data file path


In [7]:
base_path = r'/Volumes/SSD1/RawData1/Redesigned_Plastics/May2025/2025_05_Anjani/OneD_integrated_WAXS_01'
samp_folder = 'linkam'
match_partial = True
scaling_factor = 1.0
plot = False

## LDPE-master tracker

In [8]:

# List of (keyword, background_filename, subtract_azimuthal, subtract_chi, use_sample_as_background, bkg_folder, background_keyword)
samples = [

   #('Run33_f_P3HB_07_50rpm_80C_01', 's00000', True, True, True,'', ''),

    ('Run8_LDPE_30C_50ums_scan001', '', True, True, True,'', ''),
    ('Run5_LDPE_50C_50ums_scan001', '', True, True, True,'', ''),
    ('Run6_LDPE_75C_50ums_scan001', '', True, True, True,'', ''),
    ('Run7_LDPE_100C_50ums_scan001', '', True, True, True,'', ''),

    ('Run8_LDPE_30C_50ums_scan002', '', True, True, True,'', ''),
    ('Run5_LDPE_50C_50ums_scan002', '', True, True, True,'', ''),
    ('Run6_LDPE_75C_50ums_scan002', '', True, True, True,'', ''),
    ('Run7_LDPE_100C_50ums_scan002', '', True, True, True,'', ''),


    ('Run8_LDPE_30C_50ums_scan003', '', True, True, True,'', ''),
    ('Run5_LDPE_50C_50ums_scan003', '', True, True, True,'', ''),
    ('Run6_LDPE_75C_50ums_scan003', '', True, True, True,'', ''),
    ('Run7_LDPE_100C_50ums_scan003', '', True, True, True,'', ''),


  #"Run8_LDPE_30C",
   # "Run5_LDPE_50C",
    #"Run6_LDPE_75C",
   # "Run7_LDPE_100C",
  
]

for keyword, background_filename, subtract_azimuthal, subtract_chi, use_sample_as_background, bkg_folder, background_keyword in samples:
    subtract_background_v2(
        base_path=base_path,
        samp_folder=samp_folder,
        bkg_folder=bkg_folder,
        keyword=keyword,
        background_keyword=background_keyword,
        subtract_azimuthal=subtract_azimuthal,
        subtract_chi=subtract_chi,
        scaling_factor=scaling_factor,
        plot=plot,
        use_sample_as_background=use_sample_as_background,
        background_filename=background_filename,
        match_partial=match_partial
    )


2025-05-24 15:36:49
2025-05-24 15:36:59
2025-05-24 15:38:31
2025-05-24 15:38:41
2025-05-24 15:37:09
2025-05-24 15:37:19
2025-05-24 15:37:29
2025-05-24 15:37:40
2025-05-24 15:37:50
2025-05-24 15:38:00
2025-05-24 15:38:10
2025-05-24 15:38:20

✅ Using sample file as background: Run8_LDPE_30C_50ums_scan001_frame_00000 for azimuthal_data

Subtracting background from sample:
  Sample file     : Run8_LDPE_30C_50ums_scan001_frame_00000
  Background file : Run8_LDPE_30C_50ums_scan001_frame_00000

Average I in q range 8-10: 4.106498718261719
Average I in background q range 8-10: 4.106498718261719
Scaling factor for azimuthal data: 0
Subtracting background from sample:
  Sample file     : Run8_LDPE_30C_50ums_scan001_frame_00001
  Background file : Run8_LDPE_30C_50ums_scan001_frame_00000

Average I in q range 8-10: 4.161262035369873
Average I in background q range 8-10: 4.106498718261719
Scaling factor for azimuthal data: 0
Subtracting background from sample:
  Sample file     : Run8_LDPE_30C_50um

## LDPE-Gowda

In [ ]:

# List of (keyword, background_filename, subtract_azimuthal, subtract_chi, use_sample_as_background, bkg_folder, background_keyword)
samples = [

   #('Run33_f_P3HB_07_50rpm_80C_01', 's00000', True, True, True,'', ''),

    ('Run17_LDPE2_30C_50ums_scan001', '', True, True, True,'', ''),
    ('Run14_LDPE2_50C_50ums_scan001', '', True, True, True,'', ''),
    ('Run15_LDPE2_75C_50ums_scan001', '', True, True, True,'', ''),
    ('Run16_LDPE2_100C_50ums_scan001', '', True, True, True,'', ''),

    ('Run17_LDPE2_30C_50ums_scan002', '', True, True, True,'', ''),
    ('Run14_LDPE2_50C_50ums_scan002', '', True, True, True,'', ''),
    ('Run15_LDPE2_75C_50ums_scan002', '', True, True, True,'', ''),
    ('Run16_LDPE2_100C_50ums_scan002', '', True, True, True,'', ''),

    ('Run17_LDPE2_30C_50ums_scan003', '', True, True, True,'', ''),
    ('Run14_LDPE2_50C_50ums_scan003', '', True, True, True,'', ''),
    ('Run15_LDPE2_75C_50ums_scan003', '', True, True, True,'', ''),
    ('Run16_LDPE2_100C_50ums_scan003', '', True, True, True,'', ''),
  


    #"Run17_LDPE_30C",
    #"Run14_LDPE_50C",
    #"Run15_LDPE_75C",
    #"Run16_LDPE_100C",
  
]

for keyword, background_filename, subtract_azimuthal, subtract_chi, use_sample_as_background, bkg_folder, background_keyword in samples:
    subtract_background_v2(
        base_path=base_path,
        samp_folder=samp_folder,
        bkg_folder=bkg_folder,
        keyword=keyword,
        background_keyword=background_keyword,
        subtract_azimuthal=subtract_azimuthal,
        subtract_chi=subtract_chi,
        scaling_factor=scaling_factor,
        plot=plot,
        use_sample_as_background=use_sample_as_background,
        background_filename=background_filename,
        match_partial=match_partial
    )


2025-05-24 21:19:45
2025-05-24 21:19:55
2025-05-24 21:21:27
2025-05-24 21:20:05
2025-05-24 21:20:15
2025-05-24 21:20:25
2025-05-24 21:20:36
2025-05-24 21:20:46
2025-05-24 21:20:56
2025-05-24 21:21:06
2025-05-24 21:21:17

✅ Using sample file as background: Run17_LDPE2_30C_50ums_scan001_frame_00000 for azimuthal_data

Subtracting background from sample:
  Sample file     : Run17_LDPE2_30C_50ums_scan001_frame_00000
  Background file : Run17_LDPE2_30C_50ums_scan001_frame_00000

Average I in q range 8-10: 5.722381591796875
Average I in background q range 8-10: 5.722381591796875
Scaling factor for azimuthal data: 0
Subtracting background from sample:
  Sample file     : Run17_LDPE2_30C_50ums_scan001_frame_00001
  Background file : Run17_LDPE2_30C_50ums_scan001_frame_00000

Average I in q range 8-10: 5.69441032409668
Average I in background q range 8-10: 5.722381591796875
Scaling factor for azimuthal data: 0
Subtracting background from sample:
  Sample file     : Run17_LDPE2_30C_50ums_scan001

FileNotFoundError: No matching HDF5 files found in /Volumes/SSD1/RawData1/Redesigned_Plastics/May2025/2025_05_Anjani/OneD_integrated_WAXS_01/linkam with pattern *Run14_LDPE2_50C_50ums_scan001*_azimuthal_data.h5

## PHPD

In [12]:

from email.mime import base


# List of (keyword, background_filename, subtract_azimuthal, subtract_chi, use_sample_as_background, bkg_folder, background_keyword)
samples = [

   #('Run33_f_P3HB_07_50rpm_80C_01', 's00000', True, True, True,'', ''),

    ('Run9_PHPD_30C_50ums_scan001', '', True, True, True,'', ''),
    ('Run10_PHPD_45C_50ums_scan001', '', True, True, True,'', ''),
    ('Run11_PHPD_60C_50ums_scan001', '', True, True, True,'', ''),
    ('Run12_PHPD_75C_50ums_scan001', '', True, True, True,'', ''),

    ('Run9_PHPD_30C_50ums_scan002', '', True, True, True,'', ''),
    ('Run10_PHPD_45C_50ums_scan002', '', True, True, True,'', ''),
    ('Run11_PHPD_60C_50ums_scan002', '', True, True, True,'', ''),
    ('Run12_PHPD_75C_50ums_scan002', '', True, True, True,'', ''),
    
    ('Run9_PHPD_30C_50ums_scan003', '', True, True, True,'', ''),
    ('Run10_PHPD_45C_50ums_scan003', '', True, True, True,'', ''),
    ('Run11_PHPD_60C_50ums_scan003', '', True, True, True,'', ''),
    ('Run12_PHPD_75C_50ums_scan003', '', True, True, True,'', ''),

    


    #"Run9_PHPD_30C",
     #"Run10_PHPD_45C",
     #"Run11_PHPD_60C",
     #"Run12_PHPD_75C",
  
]

for keyword, background_filename, subtract_azimuthal, subtract_chi, use_sample_as_background, bkg_folder, background_keyword in samples:
    subtract_background_v2(
        base_path=base_path,
        samp_folder=samp_folder,
        bkg_folder=bkg_folder,
        keyword=keyword,
        background_keyword=background_keyword,
        subtract_azimuthal=subtract_azimuthal,
        subtract_chi=subtract_chi,
        scaling_factor=scaling_factor,
        plot=plot,
        use_sample_as_background=use_sample_as_background,
        background_filename=background_filename,
        match_partial=match_partial
    )


2025-05-24 16:14:29
2025-05-24 16:14:39
2025-05-24 16:16:11
2025-05-24 16:16:21
2025-05-24 16:16:32
2025-05-24 16:16:42
2025-05-24 16:16:52
2025-05-24 16:17:02
2025-05-24 16:14:49
2025-05-24 16:15:00
2025-05-24 16:15:10
2025-05-24 16:15:20
2025-05-24 16:15:30
2025-05-24 16:15:41
2025-05-24 16:15:51
2025-05-24 16:16:01

✅ Using sample file as background: Run9_PHPD_30C_50ums_scan001_frame_00000 for azimuthal_data

Subtracting background from sample:
  Sample file     : Run9_PHPD_30C_50ums_scan001_frame_00000
  Background file : Run9_PHPD_30C_50ums_scan001_frame_00000

Average I in q range 8-10: 6.543511390686035
Average I in background q range 8-10: 6.543511390686035
Scaling factor for azimuthal data: 0
Subtracting background from sample:
  Sample file     : Run9_PHPD_30C_50ums_scan001_frame_00001
  Background file : Run9_PHPD_30C_50ums_scan001_frame_00000

Average I in q range 8-10: 6.649869441986084
Average I in background q range 8-10: 6.543511390686035
Scaling factor for azimuthal da